# Incremental Workflow

Build cache once → merge with delta every session → instant load.

In [ ]:
from vulnflow import VulnDataset
from vulnflow.transforms.flatten import CVSSFlattenTransform, CWEPrimaryTransform, CPEPrimaryTransform
import tempfile
from pathlib import Path
import numpy as np

## Setup demo cache

Simulating your full NVD cache.

In [ ]:
demo_dir = Path(tempfile.mkdtemp())
cache_path = demo_dir / "cache.parquet"

VulnDataset.from_nvd("../data/nvdcve-2.0-Modified.json").transform([
    CVSSFlattenTransform(),
    CWEPrimaryTransform(),
    CPEPrimaryTransform(),
]).to_parquet(cache_path)

## Create delta with updated CVE

Simulating a modified CVE from NVD's Modified feed.

In [ ]:
# Get a CVE from cache to modify
cache_df = VulnDataset.from_parquet(cache_path).to_pandas()
sample_cve_id = cache_df.iloc[0]['cve_id']

# Create delta with updated score
delta_ds = VulnDataset()
delta_ds._data = {
    "cve_id": np.array([sample_cve_id, "CVE-9999-NEW"], dtype=object),
    "description": np.array(["UPDATED: Score changed", "NEW: Just disclosed"], dtype=object),
    "cvss_base_score": np.array([10.0, 7.5], dtype=object),
    "cvss_base_severity": np.array(["CRITICAL", "HIGH"], dtype=object),
}

delta_path = demo_dir / "delta.parquet"
delta_ds.to_parquet(delta_path)

## Merge cache + delta

Delta overwrites cache automatically (upsert semantics).

In [ ]:
df = VulnDataset.from_parquet([str(cache_path), str(delta_path)]).to_pandas()
len(df)

## Verify upsert worked

In [ ]:
# Updated CVE has new score
updated = df[df['cve_id'] == sample_cve_id].iloc[0]
updated[['cve_id', 'cvss_base_score', 'description']]

In [ ]:
# New CVE exists
new = df[df['cve_id'] == 'CVE-9999-NEW'].iloc[0]
new[['cve_id', 'cvss_base_score', 'description']]

## Production pattern

```python
# ONE-TIME: Build cache from full feed
VulnDataset.from_nvd("nvdcve-2.0-Recent.json") \
    .transform([CVSSFlattenTransform(), ...]) \
    .to_parquet("cache.parquet")

# EVERY SESSION: Download Modified feed, merge
VulnDataset.from_nvd("nvdcve-2.0-Modified.json") \
    .transform([CVSSFlattenTransform(), ...]) \
    .to_parquet("delta.parquet")

df = VulnDataset.from_parquet(["cache.parquet", "delta.parquet"]).to_pandas()
```

**Performance:** 300K cache + 2K delta = ~0.2 seconds

In [ ]:
# Cleanup
import shutil
shutil.rmtree(demo_dir)